<a href="https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

**Lane:** CTR / Engagement Opportunity Scoring

This notebook checks that the features used to score content are available before prediction and do not contain future outcomes, label-derived fields, client-identifying information, private queries, or other leakage.

**Prediction setup:** February 2026 features → March 2026 future CTR outcome.

## 1. Build the feature vector

The feature vector is intentionally small and uses only information available by the end of February 2026:

- `feb_impressions`
- `feb_clicks`
- `feb_ctr`
- `feb_avg_position`

IDs are used only for grouping/joining and are not model features.

In [3]:
import os
import duckdb
import pandas as pd

os.makedirs("work/outputs", exist_ok=True)

# Use the READ token from the Colab Secrets panel.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

# Use a DuckDB session variable so the token is not interpolated into SQL text.
if HF_TOKEN:
    con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")

print("Warehouse paths configured: February features + March outcome")

feature_sql = f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions,0) ELSE 0 END) AS feb_impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks,0) ELSE 0 END) AS feb_clicks,
        CASE
            WHEN SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions,0) ELSE 0 END) > 0
            THEN 100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks,0) ELSE 0 END)
                 / SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions,0) ELSE 0 END)
            ELSE NULL
        END AS feb_ctr,
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE AND gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        / NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE AND gsc_avg_position > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ), 0
        ) AS feb_avg_position
    FROM read_parquet('{FEB}')
    WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
    GROUP BY 1,2
)
SELECT * FROM feb
WHERE feb_impressions >= 100
"""

features = con.sql(feature_sql).df()
FEATURES = ["feb_impressions", "feb_clicks", "feb_ctr", "feb_avg_position"]

print("Feature vector:", FEATURES)
print("Rows eligible for the feature table:", len(features))
print(features[FEATURES].head())

Warehouse paths configured: February features + March outcome


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position']
Rows eligible for the feature table: 80322
   feb_impressions  feb_clicks   feb_ctr  feb_avg_position
0            299.0         0.0  0.000000         12.489933
1            733.0         6.0  0.818554          6.316508
2            514.0         0.0  0.000000          9.966926
3           2931.0         3.0  0.102354         41.814739
4            970.0         2.0  0.206186         10.307216


## 2. Feature notes

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `feb_impressions` | February Google Search impressions | Rows are restricted to measured impressions ≥100 | End of Feb |
| `feb_clicks` | February Google Search clicks | Missing GSC values contribute zero only when GSC is explicitly available | End of Feb |
| `feb_ctr` | February clicks / impressions, in percentage units | Null when impressions are zero; eligible rows require ≥100 impressions | End of Feb |
| `feb_avg_position` | Impression-weighted February average position; position 0 is treated as unavailable | Null when no valid position observations exist | End of Feb |

No March information is used to construct the feature vector.

In [4]:
print("Feature dtypes:")
print(features[FEATURES].dtypes)
print("\nMissing values:")
print(features[FEATURES].isna().sum())
print("\nAll feature columns are pre-March features:", all(c.startswith("feb_") for c in FEATURES))

Feature dtypes:
feb_impressions     float64
feb_clicks          float64
feb_ctr             float64
feb_avg_position    float64
dtype: object

Missing values:
feb_impressions     0
feb_clicks          0
feb_ctr             0
feb_avg_position    1
dtype: int64

All feature columns are pre-March features: True


## 3. The leakage hunt

The main attacks are:

1. Search for obvious future/label-derived fields in the candidate feature list.
2. Confirm that March is not queried when constructing February features.
3. Confirm that known label-derived fields such as `trend_direction` and `trend_pct` are excluded.
4. Confirm that identifiers are not included as predictive features.
5. Check that the future target is not accidentally present in the feature frame.

In [5]:
candidate_fields = FEATURES + [
    "trend_direction", "trend_pct", "future_ctr",
    "march_clicks", "march_impressions",
    "client_id", "content_id"
]

label_or_future = [
    c for c in candidate_fields
    if any(term in c.lower() for term in ["future", "march", "trend"])
]

id_fields = [c for c in candidate_fields if c in ["client_id", "content_id"]]

print("Potential future / label-derived fields found among candidate fields:", label_or_future)
print("Identifier fields found among candidate fields:", id_fields)
print("Final model feature list:", FEATURES)
print("Leakage-prone fields are excluded from the final feature vector.")

Potential future / label-derived fields found among candidate fields: ['trend_direction', 'trend_pct', 'future_ctr', 'march_clicks', 'march_impressions']
Identifier fields found among candidate fields: ['client_id', 'content_id']
Final model feature list: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position']
Leakage-prone fields are excluded from the final feature vector.


### Explicit leakage check

`trend_direction` and `trend_pct` are label-derived/leakage-prone in this warehouse and are therefore excluded. March clicks, March impressions, and March CTR belong to the future outcome window and are also excluded from the features.

Client/content IDs remain available only for grouping, joining, and evaluation; they are never passed to the model.

In [6]:
assert "trend_direction" not in FEATURES
assert "trend_pct" not in FEATURES
assert "future_ctr" not in FEATURES
assert "march_clicks" not in FEATURES
assert "march_impressions" not in FEATURES
assert "client_id" not in FEATURES
assert "content_id" not in FEATURES

print("PASS: no label-derived, future-window, or identifier fields are in the final feature vector.")

PASS: no label-derived, future-window, or identifier fields are in the final feature vector.


## 4. What I excluded and why

- **`trend_direction`** — label-derived; would leak the outcome.
- **`trend_pct`** — label-derived/future-derived; would leak the outcome.
- **March clicks / impressions / CTR** — belong to the future outcome window.
- **`future_ctr`** — the outcome itself; never a feature.
- **Client/content IDs** — pseudonymous identifiers used only for grouping and joins.
- **Private query text / query-level identifiers** — not needed for this public-safe lane.
- **Other post-February measurements** — unavailable at the prediction moment and therefore excluded.

The resulting feature vector is deliberately restricted to measured February search-performance signals.

## Self-check

- [x] Feature vector is explicitly defined.
- [x] Feature availability is before the prediction outcome.
- [x] Leakage-prone fields are tested and excluded.
- [x] IDs are not model features.
- [x] Public-safe fields only.
- [ ] Run **Runtime → Run all** in Colab and save the executed notebook back to the exact repo path.